# VeriGap — Kaggle feasibility probe

Before building the full container-free benchmark, confirm the two make-or-break unknowns on Kaggle:

1. **Model serving** — can vLLM serve a Gemma 4 QAT (`w4a16 compressed-tensors`) checkpoint on this Kaggle GPU? (Kaggle T4 is sm_75 + a modern driver — the combo that only failed on an old-driver box.)
2. **Binary portability** — does a precompiled task binary from the HF store run natively on Kaggle's Linux (no Docker)?

Requires: GPU on, Internet on, and a Kaggle **Secret** named `HF_TOKEN` (your HuggingFace token, needed for the gated Gemma model + datasets).

In [ ]:
MODEL = "google/gemma-4-e4b-it-qat-w4a16-ct"
TASK  = "abishekvashok__cmatrix.5c082c6"
import subprocess, os, sys, shutil
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version,compute_cap","--format=csv"],capture_output=True,text=True).stdout)
    HAS_GPU = True
else:
    print("no GPU on this instance (CPU run) — model-serving probe will be skipped")
    HAS_GPU = False

In [ ]:
# HF token from Kaggle Secrets (Add-ons -> Secrets -> add HF_TOKEN)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded:", bool(os.environ.get("HF_TOKEN")))
except Exception as e:
    print("WARNING: no HF_TOKEN secret found — add it under Add-ons -> Secrets.")
    print("Continuing; public datasets may still download.", type(e).__name__)

## Probe 2 — does a precompiled task binary run natively on Kaggle?

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path
import stat
d = snapshot_download("eth-sri/tsb-reference_impl", repo_type="dataset",
                      allow_patterns=f"executable/{TASK}/**")
exe = Path(d)/"executable"/TASK/"executable"
print("downloaded:", exe, exe.exists())
exe.chmod(exe.stat().st_mode | stat.S_IEXEC)
print("file type:", subprocess.run(["file", str(exe)], capture_output=True, text=True).stdout.strip())
for args in (["-V"], ["-h"], ["--help"]):
    r = subprocess.run([str(exe), *args], capture_output=True, text=True, timeout=10)
    print(f"\n$ ./executable {' '.join(args)}  -> rc={r.returncode}")
    print((r.stdout or r.stderr)[:400])
    if r.returncode == 0 or (r.stdout or r.stderr):
        print("\nBINARY_RUNS_NATIVELY = True"); break
else:
    print("\nBINARY_RUNS_NATIVELY = uncertain (check errors above for missing libs / exec format)")

## Probe 1 — can vLLM serve Gemma 4 QAT on this GPU?

In [ ]:
if HAS_GPU:
    print(subprocess.run(["pip","install","-q","-U","vllm"],capture_output=True,text=True).stdout[-400:])
else:
    print("skipping vllm install (no GPU this run)")

In [ ]:
if not HAS_GPU:
    print("MODEL_SERVES = skipped (no GPU — rerun with GPU once weekly quota resets)")
else:
    try:
        from vllm import LLM, SamplingParams
        llm = LLM(model=MODEL, max_model_len=4096, gpu_memory_utilization=0.90,
                  enforce_eager=True, dtype="half")
        out = llm.chat([{"role":"user","content":"Reply with exactly: PIPELINE_OK"}],
                       SamplingParams(max_tokens=16, temperature=0))
        print("MODEL_SERVES = True")
        print("response:", out[0].outputs[0].text.strip())
    except Exception as e:
        import traceback; traceback.print_exc()
        print("\nMODEL_SERVES = False —", type(e).__name__, str(e)[:300])